In [1]:
import numpy as np

In [2]:
data = [10,20,30,40,50]

In [3]:
sample = np.random.choice(data, size=5, replace=True)
print(sample)

[10 20 40 50 20]


In [4]:
def bootstrap_sample(data):
    sample = np.random.choice(data, size=len(data), replace=True)
    return sample
    

In [5]:
data = [10, 40, 50, 40, 30]

s1 = bootstrap_sample(data)
s2 = bootstrap_sample(data)

print(s1)
print(s2)

[10 30 30 30 10]
[40 50 50 30 40]


In [6]:
samples = []

for i in range(3):
    sample = bootstrap_sample(data)
    samples.append(sample)

print(samples)    

[array([40, 30, 40, 40, 10]), array([40, 10, 50, 40, 40]), array([30, 10, 40, 50, 50])]


In [7]:
features = ["Weather", "Temperature", "Humidity", "Wind"]

In [8]:
selected_features = np.random.choice(
    features,
    size=2,
    replace=False
)

In [9]:
print(selected_features)

['Humidity' 'Temperature']


In [10]:
from collections import Counter
predictions = ["Yes", "No", "Yes", "Yes", "No"]
counts = Counter(predictions)
print(counts)
final = counts.most_common(1)[0][0]
print(final)


Counter({'Yes': 3, 'No': 2})
Yes


In [11]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

x = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print(x.head())
print(y[:5])

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2
[0 0 0 0 0]


In [12]:
print(x.shape)
print(y.shape)
print(iris.target_names)

(150, 4)
(150,)
['setosa' 'versicolor' 'virginica']


In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(x,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)                                                

In [14]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (120, 4)
X_test: (30, 4)
y_train: (120,)
y_test: (30,)


In [15]:
indices = np.random.choice(
    len(X_train),
    size=len(X_train),
    replace=True
)

In [16]:
X_sample = X_train.iloc[indices]
y_sample = y_train[indices]

In [17]:
print("Original training data:", X_train.shape)
print("Bootstrap features:", X_sample.shape)
print("Bootstrap labels:", y_sample.shape)

Original training data: (120, 4)
Bootstrap features: (120, 4)
Bootstrap labels: (120,)


In [18]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

In [19]:
print("Gini Impurity:", gini(y_sample))

Gini Impurity: 0.6656944444444445


In [20]:
print(X_train.columns)

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')


In [21]:
thresholds = np.sort(X_train["petal length (cm)"].unique())
print(thresholds)

[1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.9 3.  3.5 3.6 3.7 3.8 3.9 4.  4.1 4.2 4.3
 4.4 4.5 4.6 4.7 4.8 4.9 5.  5.1 5.2 5.3 5.4 5.5 5.6 5.7 5.8 5.9 6.  6.1
 6.4 6.6 6.7 6.9]


In [22]:
def split_data(x, y, feature, threshold):
    left_mask = x[feature] <= threshold

    right_mask = x[feature] > threshold

    x_left = x[left_mask]
    y_left = y[left_mask]

    x_right = x[right_mask]
    y_right = y[right_mask]

    return x_left, y_left, x_right, y_right

In [23]:
x_left, y_left, x_right, y_right = split_data(X_train, y_train, "petal length (cm)", 3.0
                                             )
print(len(x_left))
print(len(y_right))
      

41
79


In [24]:
print(len(X_train))
print(len(y_train))
print(X_train.shape)

120
120
(120, 4)


In [25]:
left_mask = X_train["petal length (cm)"] <= 3.0
right_mask = X_train["petal length (cm)"] > 3.0

print(left_mask.sum())
print(right_mask.sum())
print((left_mask & right_mask).sum())

41
79
0


In [26]:
#Calculate Gini impurity for each child
gini_left = gini(y_left)
gini_right = gini(y_right)

print("left Gini:", gini_left)
print("Right Gini:", gini_right)

left Gini: 0.04759071980963725
Right Gini: 0.4999198846338727


In [29]:
#Calculate weighted Gini
def weighted_gini(y_left, y_right):
    total = len(y_left) + len(y_right)

    left_weight = len(y_left) / total
    right_weight = len(y_right) / total

    return (left_weight * gini(y_left) +
            right_weight * gini(y_right))

In [30]:
weighted_gini(y_left, y_right)

np.float64(0.34537408665225894)

In [33]:
thresholds = np.sort(
    X_train["petal length (cm)"].unique()
)

print(len(thresholds))

40


In [34]:
values = [1.4, 1.5, 1.6]

midpoints = [
    (values[i] + values[i+1]) / 2
    for i in range(len(values)-1)
]

print(midpoints)

[1.45, 1.55]


In [35]:
#Test all thresholds
thresholds = np.sort(X_train["petal length (cm)"].unique())

midpoints = [
    (thresholds[i] + thresholds[i+1]) / 2
    for i in range(len(thresholds)-1)
]

In [36]:
results = []
for threshold in midpoints:
    X_left, y_left, X_right, y_right = split_data(
        X_train, y_train, "petal length (cm)", threshold
    )
    score = weighted_gini(y_left, y_right)
    results.append((threshold, score))

In [48]:
print(results[:5])

[(np.float64(1.15), np.float64(0.6610644257703081)), (np.float64(1.25), np.float64(0.6553672316384181)), (np.float64(1.35), np.float64(0.6253687315634218)), (np.float64(1.45), np.float64(0.5490196078431372)), (np.float64(1.55), np.float64(0.4542124542124542))]


In [49]:
#find the best split
best_split = min(results, key=lambda x: x[1])
print(best_split)

(np.float64(2.45), np.float64(0.3333333333333333))


In [38]:
from sklearn.ensemble import RandomForestClassifier

In [40]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    random_state=42
)

In [42]:
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

In [44]:
from sklearn.metrics import accuracy_score, confusion_matrix
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

0.9
[[10  0  0]
 [ 0  9  1]
 [ 0  2  8]]


In [45]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.82      0.90      0.86        10
           2       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [47]:
importance = rf.feature_importances_

for feature, score in zip(x.columns, importance):
    print(feature, ":", score)

sepal length (cm) : 0.11634850540739942
sepal width (cm) : 0.014999733101994138
petal length (cm) : 0.43146640568097017
petal width (cm) : 0.4371853558096363
